RNN - Erro dos pesos computados e usado somente durante a iteração

In [2]:
import numpy as np
from numpy import linalg as LA
import pandas as pd
import operator as op
import ipynbname
import math
import matplotlib.cm as cm
import optuna
from optuna.samplers import RandomSampler
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_pareto_front
from optuna.importance import get_param_importances
from optuna.exceptions import TrialPruned
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import matplotlib as mpl
#from Testing.RTLO import *
from Functions.RLS import *
from Functions.Graphs import *
from sklearn.metrics import root_mean_squared_error as RMSE
from sklearn.metrics import mean_absolute_percentage_error as MAPE
FileName = ipynbname.name()

df = pd.read_csv(r'Dataset\Bearing1_1.csv')
sig = df['PC1'].values

def prepare_data(sig, n, m):
    X, Y = [], []
    # O shift (s) é calculado para alinhar o final de Y com a predição futura
    # Seguindo sua lógica: se n=4, m=3 -> Y começa no índice 2 (hi_3)
    s = n - m + 1 
    
    for i in range(len(sig) - n - 1):
        X.append(sig[i : i + n])
        Y.append(sig[i + s : i + s + m])
        
    return np.array(X), np.array(Y)


In [ ]:
def resize_series(data, m):
    """
    Interpola um array de tamanho n para o tamanho m (m > n).
    """
    n = len(data)
    x_old = np.linspace(0, 1, n)
    x_new = np.linspace(0, 1, m)
    
    return np.interp(x_new, x_old, data)

s1 = sig.copy()
s2 = resize_series(s1,1000)

t1 = np.linspace(0,1,len(s1))
t2 = np.linspace(0,1,len(s2))

PlotSeriesPLY(xSeries=[t1,t2],ySeries=[s1,s2],w=1000)

In [96]:
import numpy as np
import matplotlib.pyplot as plt

def Activation(x):
    return np.tanh(x)

def dActivation(x):
    return 1/np.cosh(10*np.tanh(x/10))**2  # the tanh prevents oveflow


class RNN:

    def __init__(self, n_in, n_rec, n_out, h0, tau_m=10):
        np.random.seed(42)
        self.n_in = n_in
        self.n_rec = n_rec
        self.n_out = n_out
        self.hi = np.zeros(n_rec)
        self.hf = np.zeros(n_rec)
        self.h0 = h0
        self.tau_m = tau_m

        # Initialize weights:
        self.w_in = 0.1*(np.random.rand(n_rec, n_in) - 1)
        self.w_rec = 1.5*np.random.randn(n_rec, n_rec)/n_rec**0.5
        self.w_out = 0.1*(2*np.random.rand(n_out, n_rec) - 1)/n_rec**0.5

        # Random error feedback matrix:
        self.b = np.random.randn(n_rec, n_out)/n_out**0.5
        self.k = 1


    def run_trial(self, x, yR, eta=[0.1, 0.1, 0.1]):

        k = 1
        [eta3, eta2, eta1] = eta  # learning rates for w_in, w_rec, and w_out
        t_max = np.shape(x)[0]  # number of timesteps
        dw_in, dw_rec, dw_out = 0, 0, 0  # changes to weights
        dw_in2, dw_rec2, dw_out2 = 0, 0, 0  # changes to weights

        u = np.zeros((t_max, self.n_rec))  # input (feedforward plus recurrent)
        h = np.zeros((t_max, self.n_rec))  # time-dependent RNN activity vector
        h[0] = self.h0  # initial state
        y = np.zeros((t_max, self.n_out))  # RNN output
        err = np.zeros((t_max, self.n_out))  # readout error
        p = np.zeros((self.n_rec, self.n_rec))
        q = np.zeros((self.n_rec, self.n_in))

        for jj in range(self.n_rec):
            q[jj, :] = dActivation(u[0, jj])*x[0,:]/self.tau_m

        for tt in range(t_max-1):
            u[tt+1] = np.dot(self.w_rec, h[tt]) + np.dot(self.w_in, x[tt+1])
            h[tt+1] = h[tt] + (-h[tt] + Activation(u[tt+1]))/self.tau_m
            y[tt+1] = np.dot(self.w_out, h[tt+1])
            err[tt+1] = yR[tt+1] - y[tt+1]  
            '''print('x:',x[tt+1])
            print('hi:',h[tt])
            print('hf:',h[tt+1])
            print('y:',y[tt+1])
            print('err:',err[tt+1])'''
            #print('real:',yR[tt+1])
            #print('pred:',y[tt+1])

            p = (1-1/self.tau_m)*p
            q = (1-1/self.tau_m)*q
            p += np.outer(dActivation(u[tt+1,:]), h[tt,:])/self.tau_m
            q += np.outer(dActivation(u[tt+1,:]), x[tt,:])/self.tau_m


            dw_out = eta1 * np.outer(err[tt+1], h[tt+1])
            dw_rec = eta2 * np.outer(np.dot(self.b, err[tt+1]),np.ones(self.n_rec))*p
            dw_in  = eta3 * np.outer(np.dot(self.b, err[tt+1]),np.ones(self.n_in))*q
            
            self.w_out = self.w_out +  dw_out
            self.w_rec = self.w_rec +  dw_rec
            self.w_in = self.w_in +  dw_in
            k = k + 1

        return y, h, u
    
    def PredSingle(self,x):
        u = np.dot(self.w_rec, self.hi) + np.dot(self.w_in, x)
        hf = self.hi + (-self.hi + Activation(u))/self.tau_m
        y = np.dot(self.w_out, hf)
        self.hi = hf
        return y
    
    def run_session(self, n_trials, x, y_, eta=[0.1, 0.1, 0.1]):

        t_max = np.shape(x)[0]  # number of timesteps
        loss_list = []
        readout_alignment = []

        # Flatten the random feedback matrix to check for feedback alignment:
        bT_flat = np.reshape((self.b).T,
            (np.shape(self.b)[0]*np.shape(self.b)[1]))
        bT_flat = bT_flat/np.linalg.norm(bT_flat)

        for ii in range(n_trials):
            y, h, u = self.run_trial(x, y_, eta)

            err = y_ - y
            loss = 0.5*np.mean(err**2)
            loss_list.append(loss)

            w_out_flat = np.reshape(self.w_out,
                (np.shape(self.w_out)[0]*np.shape(self.w_out)[1]))
            w_out_flat = w_out_flat/np.linalg.norm(w_out_flat)
            readout_alignment.append(np.dot(bT_flat, w_out_flat))
            print('\r'+str(ii+1)+'/'+str(n_trials)+'\t Err:'+str(loss), end='')

        return y, loss_list, readout_alignment

In [97]:
n_in, n_rec, n_out = 3, 3, 3 
h_init = np.zeros(n_rec)
X, yR = prepare_data(sig,n_in,n_out)

rnn = RNN(n_in, n_rec, n_out, h_init)
yP, h, u = rnn.run_trial(X[:],yR[:])


In [98]:
class RTLO:
    def __init__(self, nI,nR,nO,ηS=[0.1,0.1,0.1], τ=10):
        np.random.seed(42)
        self.k = 1
        self.act = 'tanh'
        self.nI, self.nR, self.nO = nI, nR, nO

        self.ηS = np.array(ηS)
        self.τ = τ

        self.x = np.zeros(nI)
        self.hP, self.hU, self.hL = [np.zeros(nR) for i in range(3)]

        self.pS = np.zeros((self.nR, self.nR))
        self.qS = np.zeros((self.nR, self.nI))

        self.ΔOS = np.zeros((nO, nR))
        self.ΔRS = np.zeros((nR, nR))
        self.ΔIS = np.zeros((nR, nI))
        
        self.wI = 0.1*(np.random.rand(n_rec, n_in) - 1)
        self.wR = 1.5*np.random.randn(n_rec, n_rec)/n_rec**0.5
        self.wO = 0.1*(2*np.random.rand(n_out, n_rec) - 1)/n_rec**0.5
        self.BS = np.random.randn(n_rec, n_out)/n_out**0.5
        
        self.yP = []
        

    def fit(self,xP,yR):

        η1,η2,η3 = self.ηS        

        uS = self.wR @ self.hP + self.wI @ xP
        hP = self.hP + (-self.hP + Activation(uS))/self.τ
        yP = self.wO @ hP
        eS = yR-yP

        '''print('x:',xP)
        print('hi:',self.hP)
        print('hf:',hP)
        print('yr:',yR)
        print('yp:',yP)
        print('err:',eS)'''

        self.yP.append(yP)

        self.pS = np.outer(dActivation(uS),self.hP)/self.τ + (1-1/self.τ)*self.pS
        self.qS = np.outer(dActivation(uS),self.x)/self.τ + (1-1/self.τ)*self.qS

        δOS = η1*np.outer(eS,hP)
        δRS = η2*np.outer((self.BS@eS),np.ones(self.nR))*self.pS
        δIS = η3*np.outer(np.dot(self.BS, eS),np.ones(self.nI))*self.qS

        self.wI = self.wI + δIS
        self.wR = self.wR + δRS
        self.wO = self.wO + δOS

        self.hP = hP
        self.x = xP

        
    

In [99]:
nI, nR, nO = 3, 3, 3 
X, yR = prepare_data(sig,nI,nO)

rnn = RTLO(nI,nR,nO)

for i in range(len(X)):
    if i == 0:
        rnn.x = X[i]
    else:
        rnn.fit(X[i],yR[i])
yP2 = rnn.yP

In [100]:
print(yR[-1])
print(yP[-1])
print(yP2[-1])

[0.21406503 0.19205265 0.1757025 ]
[0.22330228 0.21520652 0.20834748]
[0.22368435 0.2155944  0.20872985]
